# Notebook 03: DPO Trials (5 runs)

**Inputs:**
- `results/sft_winner.json` — written by NB02
- `models/sft_<winner>/` — LoRA adapter from winning SFT trial
- HuggingFace dataset: `trl-lib/ultrafeedback_binarized` (auto-downloaded)
- `data/test_prompts.json` and `data/gold_answers.json`

**Outputs:** `results/dpo_<trial>.json` × 5, `results/dpo_winner.json`, `models/dpo_<trial>/` × 5


## Step 1: Setup

In [ ]:
import os, sys, json, time
from pathlib import Path
import torch

KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path("/kaggle/working/daa-helper") if KAGGLE else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

if KAGGLE:
    import subprocess
    for pkg in ["transformers>=4.45.0","peft>=0.13.0","trl>=0.11.0",
                "bitsandbytes>=0.44.0","accelerate>=1.0.0","datasets>=3.0.0",
                "sacrebleu","bert-score"]:
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)

from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

HF_USERNAME = "hashirilyas18"

from utils.io_helpers import load_json, save_trial_result, list_existing_results
from utils.evaluation import run_inference_on_prompts, evaluate_responses, free_memory
print("Setup complete.")


In [ ]:
!pip install --upgrade transformers huggingface_hub


## Step 2: Load winner info, prompts, gold answers

In [ ]:
winner_info  = load_json("sft_winner.json",   base_dir="results")
prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data    = load_json("gold_answers.json", base_dir="data")
dpo_config   = load_json("dpo_trials.json",   base_dir="configs")

SFT_WINNER_NAME = winner_info["winning_trial"]
SFT_WINNER_DIR  = str(PROJECT_ROOT / "models" / f"sft_{SFT_WINNER_NAME}")
assert Path(SFT_WINNER_DIR).exists(), f"SFT adapter not found: {SFT_WINNER_DIR}"

prompts      = [p["prompt"]      for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]
dpo_trials   = dpo_config["trials"]

print(f"SFT winner: {SFT_WINNER_NAME}")
print(f"DPO trials: {len(dpo_trials)}")


## Step 3: Load DPO dataset from HuggingFace

We use `trl-lib/ultrafeedback_binarized` — a large, high-quality preference dataset
with `chosen` and `rejected` response pairs already formatted for DPO.
We subsample to 2000 pairs to keep training fast.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

BASE_MODEL = "TinyLlama/TinyLlama_v1.1"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading trl-lib/ultrafeedback_binarized ...")
raw = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
raw = raw.shuffle(seed=42).select(range(min(800, len(raw))))
print(f"Loaded {len(raw)} preference pairs")
print(f"Columns: {raw.column_names}")

# The dataset already has 'prompt', 'chosen', 'rejected' columns
# TRL's DPOTrainer accepts these directly — no reformatting needed
split   = int(0.9 * len(raw))
dpo_train_ds = raw.select(range(split))
dpo_eval_ds  = raw.select(range(split, len(raw)))

print(f"Train: {len(dpo_train_ds)}  Eval: {len(dpo_eval_ds)}")


## Step 4: Training and evaluation functions

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

def load_sft_winner():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb,
        device_map="auto", trust_remote_code=True,
        use_safetensors=False, torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, SFT_WINNER_DIR, is_trainable=True)
    return model

def run_dpo_trial(trial, train_ds, eval_ds, output_dir):
    print(f"\n{'='*60}\nDPO TRIAL: {trial['name']}\n{'='*60}")
    model = load_sft_winner()

    args = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=trial["num_train_epochs"],
        per_device_train_batch_size=trial["per_device_train_batch_size"],
        gradient_accumulation_steps=trial["gradient_accumulation_steps"],
        per_device_eval_batch_size=trial["per_device_train_batch_size"],
        learning_rate=trial["learning_rate"],
        beta=trial["beta"],
        max_length=trial["max_length"],
        max_prompt_length=trial["max_prompt_length"],
        logging_steps=5, eval_strategy="steps", eval_steps=20,
        save_strategy="no", warmup_steps=5,
        lr_scheduler_type="cosine", bf16=True,
        optim="paged_adamw_8bit", report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        processing_class=tokenizer,
    )

    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    eval_metrics = trainer.evaluate()

    train_metrics = {
        "train_loss": result.training_loss,
        "eval_loss": eval_metrics.get("eval_loss"),
        "rewards_chosen":   eval_metrics.get("eval_rewards/chosen"),
        "rewards_rejected": eval_metrics.get("eval_rewards/rejected"),
        "rewards_margin":   eval_metrics.get("eval_rewards/margins"),
        "train_runtime_seconds": elapsed,
    }
    del trainer, model
    free_memory()
    return train_metrics

def evaluate_adapter(adapter_dir):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb,
        device_map="auto", trust_remote_code=True,
        use_safetensors=False, torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    responses    = run_inference_on_prompts(model, tokenizer, prompts, max_new_tokens=512)
    eval_results = evaluate_responses(responses, gold_answers)
    samples      = [{"prompt":p,"response":r,"gold":g}
                    for p,r,g in zip(prompts,responses,gold_answers)]
    del model, base
    free_memory()
    return eval_results, samples


## Step 5: Run all 5 DPO trials

In [ ]:
existing = set(list_existing_results(stage="dpo"))
print(f"Already done: {existing}")

for trial in dpo_trials:
    fname = f"dpo_{trial['name']}.json"
    if fname in existing:
        print(f"✓ Skipping {trial['name']} (already done)")
        continue

    out_dir = str(PROJECT_ROOT / "models" / f"dpo_{trial['name']}")
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    try:
        train_metrics = run_dpo_trial(trial, dpo_train_ds, dpo_eval_ds, out_dir)
        eval_results, samples = evaluate_adapter(out_dir)
        save_trial_result(trial['name'], "dpo", trial, eval_results, train_metrics, samples)

        print(f"✓ {trial['name']}")
        print(f"  BLEU: {eval_results['aggregate']['mean_bleu']:.2f}  "
              f"BERTScore: {eval_results['aggregate']['mean_bertscore_f1']:.4f}  "
              f"EvalLoss: {train_metrics['eval_loss']:.4f}")
    except Exception as e:
        import traceback; traceback.print_exc()
        free_memory()


        # --- push after every trial ---
        try:
            from kaggle_secrets import UserSecretsClient as _UGC
            _s = _UGC()
            _gt = _s.get_secret("GITHUB_TOKEN")
            _gu = _s.get_secret("GITHUB_USER")
            import subprocess as _sp
            _sp.run(["git","config","--global","user.email","you@example.com"], capture_output=True)
            _sp.run(["git","config","--global","user.name",_gu], capture_output=True)
            _sp.run(["git","-C",str(PROJECT_ROOT),"add","results/"], capture_output=True)
            _sp.run(["git","-C",str(PROJECT_ROOT),"commit","-m",f"dpo {trial['name']} result"], capture_output=True)
            _sp.run(["git","-C",str(PROJECT_ROOT),"push",
                     f"https://{_gt}@github.com/{_gu}/daa-helper.git"], capture_output=True)
            print(f"  ✓ Pushed {trial['name']} results to GitHub")
        except Exception as _pe:
            print(f"  Git push failed (non-critical): {_pe}")

print("\nAll DPO trials done.")


## Step 6: Pick winner

In [ ]:
results = []
for trial in dpo_trials:
    try:
        results.append(load_json(f"dpo_{trial['name']}.json", base_dir="results"))
    except FileNotFoundError:
        pass

results.sort(key=lambda r: (
    -r["evaluation"]["aggregate"]["combined_score"],
     r["training_metrics"].get("eval_loss") or float("inf")
))

print(f"{'Trial':<30} {'BLEU':>7} {'BERTScore':>10} {'Combined':>10} {'EvalLoss':>10}")
for r in results:
    a = r["evaluation"]["aggregate"]
    print(f"{r['trial_name']:<30} {a['mean_bleu']:>7.2f} "
          f"{a['mean_bertscore_f1']:>10.4f} {a['combined_score']:>10.4f} "
          f"{r['training_metrics'].get('eval_loss', float('nan')):>10.4f}")

winner = results[0]
print(f"\n🏆 DPO WINNER: {winner['trial_name']}")

with open(PROJECT_ROOT / "results" / "dpo_winner.json", "w") as f:
    json.dump({"winning_trial": winner['trial_name'],
               "winning_config": winner["config"],
               "winning_metrics": winner["evaluation"]["aggregate"]}, f, indent=2)

print("Saved → results/dpo_winner.json")
print("\n✓ Notebook 03 complete. Run Notebook 04 next.")


## Step 7: Push results to GitHub

In [ ]:
import subprocess

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
GITHUB_USER  = secrets.get_secret("GITHUB_USER")

cmds = [
    ["git","config","--global","user.email","you@example.com"],
    ["git","config","--global","user.name", GITHUB_USER],
    ["git","-C",str(PROJECT_ROOT),"add","results/"],
    ["git","-C",str(PROJECT_ROOT),"commit","-m","dpo results"],
    ["git","-C",str(PROJECT_ROOT),"push",
     f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)

print("✓ Results pushed to GitHub.")


In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

# Push winning DPO adapter to HF Hub
winner_adapter_dir = str(PROJECT_ROOT / "models" / f"dpo_{winner['trial_name']}")
api = HfApi()
api.create_repo(f"{HF_USERNAME}/daa-helper-tinyllama-dpo-winner", repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    folder_path=winner_adapter_dir,
    repo_id=f"{HF_USERNAME}/daa-helper-tinyllama-dpo-winner",
    repo_type="model",
    commit_message=f"DPO winner: {winner['trial_name']}"
)
print(f"Pushed DPO adapter to HF Hub: {HF_USERNAME}/daa-helper-tinyllama-dpo-winner")

# Push results to GitHub
import subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
GITHUB_USER  = secrets.get_secret("GITHUB_USER")

cmds = [
    ["git","config","--global","user.email","you@example.com"],
    ["git","config","--global","user.name", GITHUB_USER],
    ["git","-C",str(PROJECT_ROOT),"add","results/"],
    ["git","-C",str(PROJECT_ROOT),"commit","-m","dpo results"],
    ["git","-C",str(PROJECT_ROOT),"push",
     f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)

print("Done. Notebook 03 complete. Run Notebook 04 next.")
